# Import

In [1]:
import numpy as np

In [2]:

# This an easy test example.
"""
group_one_column = ["X", "X", "X", "Y", "Y", "Z"]
group_two_column = ["Y", "Z", "Q", "Z", "Q", "Q"]
p_values = [0.1, 0.06, 0.01, 0.06, 0.01, 0.04]
true_cld = {
    "X": "a",
    "Y": "a",
    "Z": "a",
    "Q": "b"
}

"""
"""
# This is a more complex test example.
group_one_column = ["X", "X", "Y"]
group_two_column = ["Y", "Z", "Z"]
p_values = [0.1, 0.04, 0.1]
true_cld = {
    "X": "a",
    "Y": "ab",
    "Z": "b"
}
"""

# This is a more complex test example.

group_one_column = ["U", "U", "U", "U", "U", "V", "V", "V", "V", "W", "W", "W", "X", "X", "Y"]
group_two_column = ["V", "W", "X", "Y", "Z", "W", "X", "Y", "Z", "X", "Y", "Z", "Y", "Z", "Z"]
p_values =         [0,   1,   0,   1,    1,   0,  1,   0,    0,  0,   1,   1,   0,    1,   1]
true_cld = {
    "U": "c",
    "V": "a",
    "W": "c",
    "X": "ab",
    "Y": "c",
    "Z": "bc"
}



# Prepare things 

In [3]:
alpha = 0.05

def calc_big_H():
    capital_H = list()
    for (group_one, group_two, p_value) in zip(group_one_column, group_two_column, p_values):
        if p_value < alpha:
            capital_H.append((group_one, group_two))
            
    return capital_H

def find_unique_groups():

    all_unique_groups = group_one_column + group_two_column
    
    unique_groups = tuple(list(dict.fromkeys(all_unique_groups))) # Ensure inmutable.
    
    return unique_groups


capital_H = calc_big_H()
capital_H
unique_groups = find_unique_groups()
unique_groups = sorted(unique_groups)
capital_H

[('U', 'V'),
 ('U', 'X'),
 ('V', 'W'),
 ('V', 'Y'),
 ('V', 'Z'),
 ('W', 'X'),
 ('X', 'Y')]

# Insert and Absorb 

In [4]:
# Insert
def insert_new_columns(M, i, j):    
    #IMPORTANT: I believe it is working, but needs checking.

    # Create a new matrix.
    new_matrix_columns = []
    idx_of_new_columns = []
    # Iterate over columns of M. (M.shape[1] is number of columns).
    for column_index in range(M.shape[1]):
        # Check whether column needs to be duplicated. 
        column_in_M = M[:, column_index]
        ith_position = column_in_M[i]
        jth_position = column_in_M[j]

        # No insertion needed.
        if (ith_position == 1 and jth_position == 0) or (ith_position == 0 and jth_position == 1):
            new_matrix_columns.append(column_in_M)
            idx_of_new_columns.append(len(new_matrix_columns) - 1)
        
        # Column needs to be duplicated if it contains 1 on both, i and j.
        else:
            # One copy must be like M, with ith position 0, and jth position 1.
            # The other copy must be like M, with ith position 1, and jth position 0.
            column_copy_one = column_in_M.copy()
            column_copy_one.put([i, j], [0, 1])

            column_copy_two = column_in_M.copy()
            column_copy_two.put([i, j], [1, 0])

            new_matrix_columns.append(column_copy_one)
            new_matrix_columns.append(column_copy_two)
            
            # Index of newly added columns.
            idx_of_new_columns.append(len(new_matrix_columns) - 2)
            idx_of_new_columns.append(len(new_matrix_columns) - 1)
    return new_matrix_columns, idx_of_new_columns

In [5]:

def absorb_columns(M, idx_of_new_columns):

    not_absorbed_cols = [] # Collects all columns that need to be kept.

    # Tracks all cols that have been absobed.
    # Avoids 
    absorbed_cols_indices = [] 
    print("----------------")
    print("M_olc", M)
    for col_one_id, col_one in enumerate(M):
        can_col_one_be_absorbed = False

        # Check each column.
        non_zero_col_one_idx = col_one.nonzero()
        # Compare against each other column
        for col_two_id, col_two in enumerate(M):
            # Skip comparison if cols are identical.
            if col_one_id == col_two_id:
                continue
            # Skip if col two has already been absorbed.
            elif col_two_id in absorbed_cols_indices: 
                continue
            # Otherwise do the comparison. 
            else: 
                non_zero_col_two_idx = col_two.nonzero()
                col_one_is_completly_in_col_two = np.in1d(non_zero_col_one_idx, non_zero_col_two_idx).all()
                
                if col_one_is_completly_in_col_two:
                    # Column one should not be kept.
                    absorbed_cols_indices.append(col_one_id)
                    can_col_one_be_absorbed = True
                    break
        
        # If we reach here, col one could not be absorbed.
        if not can_col_one_be_absorbed:
            not_absorbed_cols.append(col_one)

           
    print("Not absorbed cols: ", not_absorbed_cols)
    print("----------------")
    return not_absorbed_cols
                


def heuristic_insert_absorb(unique_groups, capital_H):
    i = 0
    # FIXME: Naming ambigous!!! Change later. 
    # 1) Generate inital treatment column.
    index_column = np.array(unique_groups) # Contains treatment. Also used to find row index.
    column_one = np.ones(len(unique_groups), dtype=np.int8).reshape(-1, 1) # One starts with a column of ones.
    M = column_one # Initial letter matrix.
    
    # 2) Iterate over significantly different pairs.
    for (group_one, group_two) in capital_H: # sig_dif = two groups that are significantly different
        print(group_one, group_two)
        # 2.1) Find indices of the groups that are significantly different.
        group_one_index = np.where(index_column == group_one)[0][0]
        group_two_index = np.where(index_column == group_two)[0][0]

        # 2.2) Insert and absorb.
        M, idx_of_new_columns = insert_new_columns(M, group_one_index, group_two_index)
        M = absorb_columns(M, idx_of_new_columns)

        # Reshape letter_matrix back to 2D array.
        M = np.array(M).T
        print(M)
   
        if (i == 1):
            pass
        i += 1

    return M

letter_matrix = heuristic_insert_absorb(unique_groups, capital_H)
letter_matrix

U V
----------------
M_olc [array([0, 1, 1, 1, 1, 1], dtype=int8), array([1, 0, 1, 1, 1, 1], dtype=int8)]
Not absorbed cols:  [array([0, 1, 1, 1, 1, 1], dtype=int8), array([1, 0, 1, 1, 1, 1], dtype=int8)]
----------------
[[0 1]
 [1 0]
 [1 1]
 [1 1]
 [1 1]
 [1 1]]
U X
----------------
M_olc [array([0, 1, 1, 1, 1, 1], dtype=int8), array([0, 0, 1, 1, 1, 1], dtype=int8), array([1, 0, 1, 0, 1, 1], dtype=int8)]
Not absorbed cols:  [array([0, 1, 1, 1, 1, 1], dtype=int8), array([1, 0, 1, 0, 1, 1], dtype=int8)]
----------------
[[0 1]
 [1 0]
 [1 1]
 [1 0]
 [1 1]
 [1 1]]
V W
----------------
M_olc [array([0, 0, 1, 1, 1, 1], dtype=int8), array([0, 1, 0, 1, 1, 1], dtype=int8), array([1, 0, 1, 0, 1, 1], dtype=int8)]
Not absorbed cols:  [array([0, 0, 1, 1, 1, 1], dtype=int8), array([0, 1, 0, 1, 1, 1], dtype=int8), array([1, 0, 1, 0, 1, 1], dtype=int8)]
----------------
[[0 0 1]
 [0 1 0]
 [1 0 1]
 [1 1 0]
 [1 1 1]
 [1 1 1]]
V Y
----------------
M_olc [array([0, 0, 1, 1, 1, 1], dtype=int8), array([0,

array([[0, 0, 1],
       [0, 1, 0],
       [0, 0, 1],
       [1, 1, 0],
       [0, 0, 1],
       [1, 0, 1]], dtype=int8)

# Sweep

In [6]:
# I think this works...
# Should be tested...
def sweep(M):
    # Go each letter (columns in the letter matrix)
    for first_column_nr, unique_letter_column in enumerate(M.T):
        # Go through each treatment in the column and check letter.
        for i_index, i_th_treat_let in enumerate(unique_letter_column):
            
            # If the letter is 0, nothing needs to be done.
            if i_th_treat_let == 0:
                continue
            # If the letter is 1, check whether it can be removed. (aka, replaced with 0)
            elif i_th_treat_let == 1:

                # Check for redundancy.
                # The ith letter can be changed in this first column from 1 to 0 if all 
                # other treatments (all jth) 
                # that share the letter with i.
                # also share another letter with i in another column
                jth_share_letter_with_ith = []

                # Go through the other treatments in the same column.
                for j_index, j_th_treat_let in enumerate(unique_letter_column):
                    # Skip if j_index is the same as i_index.
                    # Also skip if j_th_treat_let is 0.
                    if j_index == i_index or j_th_treat_let == 0:
                        continue
                    # If both, i_th_treat_let and j_th_treat_let are 1,
                    # Check if they have common letter in any other column.
                    ith_and_jth_pair_found_in_other_column = False
                    for second_column_nr, second_column in enumerate(M.T):
                        
                        # Skip if second_column_nr is the same as first_column_nr.
                        if second_column_nr == first_column_nr:
                            continue
                        else:
                            # Check if both treatments have letter 1 in any second column.
                            if second_column[i_index] == 1 and second_column[j_index] == 1:
                                ith_and_jth_pair_found_in_other_column = True
                                
                                break
                            # if second_column[i_index] == 1 and second_column[j_index] == 1:
                            #     jth_share_letter_with_ith.append(True)
                            #     print(f"Treatment {i_index} and {j_index} share letter in column {second_column_nr}.")
                            #     continue
                            # # If no common letter is found, append False.
                            # jth_share_letter_with_ith.append(False)
                    jth_share_letter_with_ith.append(ith_and_jth_pair_found_in_other_column)
                # Check if all treatment-pairs i has a redundant letter in at least one other column with each j,
                ith_letter_in_first_column_redundant = all(jth_share_letter_with_ith)
                if ith_letter_in_first_column_redundant:
                    # Set the letter to 0.
                    M[i_index, first_column_nr] = 0

    # Remove empty columns (all zeros).
    non_empty_columns = []
    for column in M.T:
        if not np.all(column == 0):
            non_empty_columns.append(column)
    M = np.array(non_empty_columns).T


    # Return the swept matrix
    return M

sweeped_matrix = sweep(letter_matrix)
sweeped_matrix


array([[0, 0, 1],
       [0, 1, 0],
       [0, 0, 1],
       [1, 1, 0],
       [0, 0, 1],
       [1, 0, 1]], dtype=int8)

# Comeup with the final letters

In [7]:
# IMPORTANT: Unchecked ChatGPT code!!!
# Calculate the final CLD from the sweeped matrix.
def calculate_cld(sweeped_matrix, unique_groups):
    cld_dict = {}
    num_letters = sweeped_matrix.shape[1]
    letters = [chr(i) for i in range(97, 97 + num_letters)]  # 'a', 'b', 'c', ...
    
    for i, group in enumerate(unique_groups):
        cld = ''
        for j in range(num_letters):
            if sweeped_matrix[i, j] == 1:
                cld += letters[j]
        cld_dict[group] = cld
    
    return cld_dict

unique_groups_sorted = sorted(unique_groups)
final_cld = calculate_cld(sweeped_matrix, unique_groups_sorted)

# Ensure each group has at least one letter: assign a new unused single-letter if empty.
used_letters = set(''.join(v for v in final_cld.values() if v))
next_ord = 97 + sweeped_matrix.shape[1]  # continue after existing letters

for group in unique_groups_sorted:
    if final_cld[group] == '':
        while chr(next_ord) in used_letters:
            next_ord += 1
        letter = chr(next_ord)
        final_cld[group] = letter
        used_letters.add(letter)
        next_ord += 1

final_cld

{'U': 'c', 'V': 'b', 'W': 'c', 'X': 'ab', 'Y': 'c', 'Z': 'ac'}

# Check whether the calculated cld is free of mistakes 


In [8]:
def verify_cld(final_cld, group_one_column, group_two_column, p_values, alpha=0.05):
    for (group_one, group_two, p_value) in zip(group_one_column, group_two_column, p_values):
        letters_one = final_cld[group_one]
        letters_two = final_cld[group_two]

        # Check if there is any common letter between the two groups.
        shared_letters = set(letters_one).intersection(set(letters_two))

        if p_value <= alpha:
            # Groups should not share any letters.
            assert shared_letters == set(), f"Groups {group_one} and {group_two} share letters {shared_letters} but should not."
        else:
            # Groups should share at least one letter.
            assert shared_letters != set(), f"Groups {group_one} and {group_two} do not share any letters but should."
    return True

verify_cld(final_cld, group_one_column, group_two_column, p_values, alpha)

True

## Some testing

In [9]:

# Create an array
arr = np.array([1, 2, 3, 1, 4, 1, 5])

# Find the indices where the value is 1
indices = np.where(arr == 1)

print(indices)  # Output: (array([0, 3, 5]),)


(array([0, 3, 5], dtype=int64),)


In [10]:
indices = np.asarray(arr ==1).nonzero()

print(indices)  # Output: (array([0, 3, 5]),)

(array([0, 3, 5], dtype=int64),)


In [11]:
new_array = np.zeros((2, 5))
new_array = np.random.rand(2, 5)
print(new_array)
print(new_array.T)

[[0.93056375 0.78716295 0.46716693 0.54395294 0.64209041]
 [0.18225607 0.74193247 0.78976574 0.14703558 0.9190966 ]]
[[0.93056375 0.18225607]
 [0.78716295 0.74193247]
 [0.46716693 0.78976574]
 [0.54395294 0.14703558]
 [0.64209041 0.9190966 ]]


In [12]:
for i in np.nditer(new_array, flags=["external_loop"], order="C"):
    print("iteation")
    print(i)

iteation
[0.93056375 0.78716295 0.46716693 0.54395294 0.64209041 0.18225607
 0.74193247 0.78976574 0.14703558 0.9190966 ]


In [13]:
for i in new_array:
    print("iteation")
    print(i)

iteation
[0.93056375 0.78716295 0.46716693 0.54395294 0.64209041]
iteation
[0.18225607 0.74193247 0.78976574 0.14703558 0.9190966 ]


In [14]:
i = 0
j = 1

if (i == 1 and j == 0) or (i == 0 and j == 1):
    print("yes")

yes
